# Chapter 8 — Sequences, Time & Order

Companion notebook for **PyTorch From Ground Up, Volume 2, Chapter 8**.
Every code block printed in the chapter, in order, runnable top to bottom.

| Chapter section | Notebook section |
|---|---|
| An image, read as a sequence | 1 |
| What order actually means | 2 |
| The margin that vanishes | 3 |
| A model that cannot count to thirty-two | 4 |
| Padding, and the mask that keeps it honest | 5 |
| Deleting order on purpose | 6 |

**This notebook runs on a CPU.** No GPU, no Colab session needed. Section 3
is the only slow part — six models for five epochs, about **15 minutes** on an
ordinary laptop. `EPOCHS` is a variable at the top of that cell.

Four things this notebook adds to what the chapter prints are listed in
`README.md`.

### Not printed in the chapter: imports and Part I's two models

The chapter assumes both models from Part I are already in hand. A notebook
cannot, so they are rebuilt here exactly as Chapters 1 and 5 defined them.

In [ ]:
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [ ]:
class SmallCNN(nn.Module):
    """Chapter 5's network, unchanged."""

    def __init__(self, n_classes=10):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(), nn.MaxPool2d(2))
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(128, n_classes))

    def forward(self, x):
        return self.head(self.body(x))


def make_mlp():
    """Chapter 1's model, the one Chapter 5 measured against."""
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(3 * 32 * 32, 512), nn.ReLU(),
        nn.Linear(512, 10))

`BagOfRows` belongs to section 6 of the chapter, but the training loop in
section 3 trains it, so a runnable notebook has to define it first. This is
the chapter's block, moved earlier and otherwise untouched.

In [ ]:
class BagOfRows(nn.Module):
    def __init__(self, n_classes=10, w=128):
        super().__init__()
        self.encode = nn.Sequential(
            nn.Linear(96, w), nn.ReLU(),
            nn.Linear(w, w), nn.ReLU())
        self.classify = nn.Linear(w, n_classes)

    def forward(self, x):
        b = x.shape[0]
        s = x.permute(0, 2, 1, 3).reshape(b, 32, 96)
        return self.classify(self.encode(s).mean(1))

## 1 · An image, read as a sequence

Same tensor, same pixels, a different story about what they are.

In [ ]:
import torch

x = torch.randn(4, 3, 32, 32)
steps = x.permute(0, 2, 1, 3).reshape(4, 32, 96)

print("as a picture ", tuple(x.shape))
print("as a sequence", tuple(steps.shape))

Output:

```
as a picture  (4, 3, 32, 32)
as a sequence (4, 32, 96)
```

## 2 · What order actually means

One permutation, drawn once from its own generator, applied identically to
every training and test image.

In [ ]:
class PermuteRows:
    def __init__(self, perm):
        self.perm = perm

    def __call__(self, x):   # x: (C, H, W)
        return x[:, self.perm, :]

g = torch.Generator().manual_seed(1234)
perm = torch.randperm(32, generator=g)
print(perm[:8].tolist())

Output:

```
[15, 9, 8, 1, 4, 12, 30, 7]
```

## 3 · The margin that vanishes

### Not printed in the chapter: the data and the training loop

The chapter prints the results table but not the loop that produced it — it
is Chapter 5's `run_epoch`, unchanged. Here it is in full so the table can be
reproduced rather than taken on trust.

**This is the slow cell.** Six configurations at five epochs is about 15
minutes on a laptop CPU. Lower `EPOCHS` for a quick look, but the accuracies
stop being comparable to the book's.

In [ ]:
EPOCHS = 5
NORM = transforms.Normalize([0.5] * 3, [0.5] * 3)
base_tf = transforms.Compose([transforms.ToTensor(), NORM])
perm_tf = transforms.Compose([transforms.ToTensor(), NORM,
                              PermuteRows(perm)])


def loaders(shuffled):
    tf = perm_tf if shuffled else base_tf
    tr = datasets.CIFAR10("data", train=True, download=True,
                          transform=tf)
    te = datasets.CIFAR10("data", train=False, download=True,
                          transform=tf)
    g = torch.Generator().manual_seed(0)
    return (DataLoader(tr, batch_size=128, shuffle=True,
                       generator=g),
            DataLoader(te, batch_size=256))


def run_epoch(model, loader, opt=None):
    training = opt is not None
    model.train(training)
    loss_sum = correct = n = 0
    for xb, yb in loader:
        with torch.set_grad_enabled(training):
            o = model(xb)
            loss = F.cross_entropy(o, yb)
        if training:
            opt.zero_grad()
            loss.backward()
            opt.step()
        loss_sum += loss.item() * yb.size(0)
        correct += (o.argmax(1) == yb).sum().item()
        n += yb.size(0)
    return loss_sum / n, correct / n

`BagOfRows` is defined in section 6; the loop below trains it too, so run
this notebook top to bottom rather than jumping straight here.

In [ ]:
def build(kind):
    torch.manual_seed(0)
    return {"cnn": SmallCNN, "mlp": make_mlp,
            "bag": BagOfRows}[kind]()


results = {}
for kind in ("cnn", "mlp", "bag"):
    for shuffled in (False, True):
        key = f"{kind}_{'shuffled' if shuffled else 'natural'}"
        model = build(kind)
        opt = torch.optim.Adam(model.parameters(), lr=1e-3)
        tr, te = loaders(shuffled)
        t0 = time.time()
        for ep in range(EPOCHS):
            run_epoch(model, tr, opt)
        _, acc = run_epoch(model, te)
        n_p = sum(p.numel() for p in model.parameters())
        results[key] = (n_p, acc)
        print(f"{key:13s} {n_p:>9,}  test {acc:.4f}"
              f"  [{time.time() - t0:.0f}s]", flush=True)

In [ ]:
gap_nat = (results["cnn_natural"][1]
           - results["mlp_natural"][1]) * 100
gap_shuf = (results["cnn_shuffled"][1]
            - results["mlp_shuffled"][1]) * 100
print(f"CNN advantage, rows in order: {gap_nat:+.1f} pts")
print(f"CNN advantage, rows shuffled: {gap_shuf:+.1f} pts")

## 4 · A model that cannot count to thirty-two

`mlp` and `cnn` below are fresh, untrained instances — the point is what the
architectures accept, not what they predict.

In [ ]:
mlp, cnn = make_mlp(), SmallCNN()

In [ ]:
short = torch.randn(1, 3, 24, 32)

try:
    mlp(short)
except RuntimeError as e:
    print("MLP:", e)

print("CNN:", tuple(cnn(short).shape))

Output:

```
MLP: mat1 and mat2 shapes cannot be multiplied (1x2304 and 3072x512)
CNN: (1, 10)
```

## 5 · Padding, and the mask that keeps it honest

In [ ]:
lengths = [32, 24, 18, 30]
seqs = [torch.randn(L, 96) for L in lengths]
T = max(lengths)

padded = torch.zeros(len(seqs), T, 96)
mask = torch.zeros(len(seqs), T, dtype=torch.bool)
for i, s in enumerate(seqs):
    padded[i, :len(s)] = s
    mask[i, :len(s)] = True

naive = padded.mean(dim=1)
real = mask.sum(1, keepdim=True)
masked = (padded * mask.unsqueeze(-1)).sum(1) / real
print((naive - masked).abs().max().item())

Output:

```
0.3979
```

## 6 · Deleting order on purpose

`BagOfRows` is defined above, next to the other two models, because section 3
trains it. Its whole trick is the `.mean(1)` over steps: averaging does not
care what order its arguments arrive in, so the model is blind to the sequence
by construction rather than by accident.

Its two rows in the table from section 3 are the control for the whole
chapter — they should agree to within a handful of images.

## Reproducing the book's numbers

The book's run: torch 2.13.0, CPU, `manual_seed(0)`, Adam lr 1e-3, batch 128
train / 256 eval, 5 epochs, CIFAR-10 normalised to mean 0.5 / std 0.5, row
permutation seeded 1234.

| Model | Rows | Params | Test |
|---|---|---|---|
| SmallCNN | in order | 94,538 | **58.4%** |
| SmallCNN | shuffled | 94,538 | **52.0%** |
| Flatten + Linear | in order | 1,578,506 | **51.4%** |
| Flatten + Linear | shuffled | 1,578,506 | **51.7%** |
| BagOfRows | in order | 30,218 | **40.6%** |
| BagOfRows | shuffled | 30,218 | **40.8%** |

The convolutional network is **+7.0 points** ahead in order and **+0.3
points** ahead shuffled. That collapse is the chapter.

Two things to expect when you run it. The CNN lands near 58.4% rather than
Chapter 5's 59.7% — same architecture, same recipe, a different
batch-shuffling stream, and worth remembering before you read anything into a
point of accuracy. And the two `BagOfRows` rows will not be *exactly* equal
even though the model is permutation-invariant: `float32` addition is not
associative, so reordering the rows reorders the sum inside `.mean()`. The
book's two runs disagree on 17 of 10,000 test images. The CNN's two runs
disagree on 636.